# Aula 03 — Visualização Exploratória de Dados

## Objetivos
1. Construir e interpretar **histogramas** e curvas de **densidade (KDE)**.
2. Construir e interpretar **boxplots** e identificar outliers.
3. Avaliar **assimetria** e **curtose** visual e numericamente.
4. Combinar múltiplos gráficos com `seaborn` para diagnóstico rápido.

---

## 1. Por que visualizar?

> *"Um gráfico vale mais que mil estatísticas resumidas."*

O **Quarteto de Anscombe** (1973) mostra 4 datasets com **mesma** média, variância,
correlação e reta de regressão — mas formas completamente diferentes. Sem gráfico,
você nunca descobriria.

Visualização exploratória (EDA) responde:
- A distribuição é simétrica?
- Há outliers?
- Há múltiplos picos (multimodalidade)?
- A escala precisa de transformação (ex.: log)?

---

## 2. Histograma

Divide os dados em **bins** (faixas) e conta as observações em cada um.

- **Poucos bins** → esconde detalhes.
- **Muitos bins** → ruído.
- Regra prática: $k = \lceil \sqrt{n} \rceil$ ou regra de Sturges $k = 1 + \log_2 n$.

## 3. Densidade (KDE)

A **Kernel Density Estimation** é uma versão "suavizada" do histograma.
Em vez de barras, coloca uma pequena curva (kernel gaussiano, em geral) sobre cada
observação e soma todas. Resultado: curva contínua que estima a densidade da população.

## 4. Boxplot

Resume a distribuição em 5 números:
- Q1, Mediana, Q3 → a "caixa".
- Whiskers (bigodes) → até $Q1 - 1{,}5 \times IQR$ e $Q3 + 1{,}5 \times IQR$.
- Pontos fora dos whiskers → **candidatos a outliers**.

## 5. Assimetria (skewness) e Curtose (kurtosis)

- **Skewness > 0** → cauda à direita (média > mediana).
- **Skewness < 0** → cauda à esquerda.
- **Skewness ≈ 0** → simétrica.
- **Kurtosis > 0 (excess)** → caudas pesadas (leptocúrtica).
- **Kurtosis < 0** → caudas leves (platicúrtica).

In [ ]:
# === SETUP PARA GOOGLE COLAB ===
# Esta célula garante que o notebook funcione tanto em ambiente local
# quanto em quem clonar o repositório direto no Colab.

import sys, os, subprocess

# 1) Instala dependências (silencioso se já existirem)
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "requests", "plotly", "statsmodels"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# 2) Se estivermos no Colab e o repo ainda não foi clonado, clona.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/220719/curso-estatistica-python.git"  # ajuste após criar o repo

if IN_COLAB and not os.path.exists("/content/curso-estatistica-python"):
    subprocess.run(["git", "clone", REPO_URL, "/content/curso-estatistica-python"], check=False)

# 3) Adiciona o diretório utils ao PYTHONPATH para importar o módulo SIDRA
BASE = "/content/curso-estatistica-python" if IN_COLAB else ".."
if BASE not in sys.path:
    sys.path.append(BASE)

# 4) Pasta de gráficos (cria se não existir)
GRAFICOS_DIR = os.path.join(BASE, "graficos")
os.makedirs(GRAFICOS_DIR, exist_ok=True)

print("✓ Ambiente pronto. Pasta de gráficos:", GRAFICOS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style="whitegrid", palette="deep")

from utils.sidra_api import obter_ipca_mensal, obter_pib_per_capita_uf

df_ipca = obter_ipca_mensal(120)  # 10 anos
df_pib  = obter_pib_per_capita_uf()

## 6. Histograma + KDE do IPCA

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Histograma com densidade (não frequência absoluta) para sobrepor a KDE
sns.histplot(df_ipca["variacao_mensal"], bins=20, kde=True,
             stat="density", color="steelblue", edgecolor="white", ax=ax)
ax.axvline(df_ipca["variacao_mensal"].mean(),   color="red",   linestyle="--", label="Média")
ax.axvline(df_ipca["variacao_mensal"].median(), color="black", linestyle=":",  label="Mediana")
ax.set_title("Distribuição do IPCA mensal (últimos 10 anos)", fontweight="bold")
ax.set_xlabel("Variação mensal (%)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula03_ipca_hist_kde.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Boxplot — IPCA vs PIB per capita

Comparando duas distribuições muito diferentes em escala.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(y=df_ipca["variacao_mensal"], color="steelblue", ax=axes[0])
axes[0].set_title("Boxplot — IPCA mensal (%)")
axes[0].set_ylabel("Variação (%)")

sns.boxplot(y=df_pib["pib_per_capita"], color="orange", ax=axes[1])
axes[1].set_title("Boxplot — PIB per capita por UF (R$)")
axes[1].set_ylabel("R$")

plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula03_boxplots.png"), dpi=150, bbox_inches="tight")
plt.show()

### Detectando outliers manualmente

In [ ]:
def detectar_outliers_iqr(serie):
    """Retorna os valores fora do intervalo [Q1 - 1.5*IQR, Q3 + 1.5*IQR]."""
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    return serie[(serie < lim_inf) | (serie > lim_sup)]

print("Outliers no IPCA mensal:")
print(detectar_outliers_iqr(df_ipca["variacao_mensal"]).values)
print("\nOutliers no PIB per capita UF:")
out_pib = detectar_outliers_iqr(df_pib["pib_per_capita"])
print(df_pib.loc[out_pib.index, ["uf", "pib_per_capita"]])

## 8. Assimetria e curtose

In [ ]:
for nome, serie in [("IPCA", df_ipca["variacao_mensal"]),
                    ("PIB per capita", df_pib["pib_per_capita"])]:
    sk = stats.skew(serie)
    ku = stats.kurtosis(serie)  # excesso de curtose (normal = 0)
    print(f"{nome:>15s} | skew = {sk:+.3f} | kurtosis = {ku:+.3f}")

print("\nInterpretação:")
print("  skew > 0  → cauda à direita")
print("  kurt > 0  → caudas mais pesadas que a Normal")

## 9. Pairplot rápido para múltiplas variáveis

In [ ]:
# Vamos construir um DataFrame com várias estatísticas do IPCA por ano
df_ipca["ano"] = df_ipca["data"].dt.year
resumo_anual = df_ipca.groupby("ano")["variacao_mensal"].agg(["mean", "std", "min", "max"]).reset_index()
resumo_anual.columns = ["ano", "media", "desvio", "min", "max"]
resumo_anual

In [ ]:
sns.pairplot(resumo_anual.drop(columns="ano"), diag_kind="kde",
             plot_kws={"alpha": 0.7})
plt.suptitle("Pairplot — estatísticas anuais do IPCA", y=1.02, fontweight="bold")
plt.savefig(os.path.join(GRAFICOS_DIR, "aula03_pairplot.png"), dpi=150, bbox_inches="tight")
plt.show()

## Exercícios

1. Refaça o histograma do IPCA usando **regra de Sturges** para definir o número de bins.
2. Identifique os meses considerados outliers e tente associá-los a eventos econômicos
   (ex.: pandemia, choques de combustíveis).
3. Compare visualmente as distribuições do IPCA em dois períodos:
   `2015-2019` vs `2020-2024`. Use boxplots lado a lado (`sns.boxplot` com `hue`).

---

**Próxima aula:** Probabilidade e Distribuições.